# **LLM Agent for Financial News Analysis: Topic Discovery on NVIDIA Coverage**

An LLM agent that explores a financial news dataset, runs the analysis through tools and explains the results. The model is NVIDIA Nemotron-3-Nano, served locally on a Colab GPU with vLLM and orchestrated with LangChain.

**Stack:** Python, pandas, LangChain, LangGraph, vLLM, NVIDIA Nemotron, scikit-learn, sentence-transformers

**Data:** [magodobolo/nvidia-news](https://www.kaggle.com/datasets/magodobolo/nvidia-news) on Kaggle (`news_v1.csv`, 1047 articles from 2015 to 2024, mostly in Portuguese)

## Business problem

Market intelligence, investor relations and equity research teams follow a constant stream of news about the companies they cover. Reading everything is slow, and keyword search only finds what the analyst already knows to look for.

The problem gets worse because a large share of the feed is not editorial content. Daily market wraps, advertising blocks and agency bylines repeat across hundreds of articles and hide the real themes.

The team needs a faster way to answer questions such as:

- What are the main themes in the news about NVIDIA?
- How much of the coverage is about the company itself, and how much is general market context?
- Which articles are templated reports, and which carry analysis?
- Can I ask for an analysis in plain language and get numbers I can trust?

## Proposed solution

An LLM agent with tools. Instead of answering from memory, the agent decides when to call each tool (pandas, semantic search, KMeans, LDA, plots), reads the result and explains it. The numbers in the answer come from the tools, not from the model.

## Objectives

1. Profile the dataset and detect quality issues before modeling.
2. Clean templated text so topics reflect editorial content.
3. Discover themes with two methods and compare them: KMeans on embeddings and LDA on 2- to 3-word phrases.
4. Let the agent name the topics and generate the plots on request.
5. Document what worked, what failed and what still needs validation.

## Success criteria

- Each topic can be explained by reading its representative headlines, not only its keywords.
- The agent calls the right tool for each question and reports tool numbers exactly.
- Templated text (advertising, daily wraps, bylines) no longer dominates the topics.
- Every limitation found is written down, including the weak silhouette score and the unstable LDA on short texts.

## Scope

In scope: the `Company == "NVIDIA"` rows of `news_v1.csv`, topic discovery, visualization and the agent workflow.

Out of scope for now: sentiment (the `nvidia_sentimental_analisis.csv` file), topic evolution over time and any investment recommendation. The analysis describes what the news says. It does not predict prices.

## Notebook outline

1. Setup: dependencies, secrets and dataset download
2. Data profiling and quality checks
3. Text cleaning: bylines, advertising, repeated sentences and daily wraps
4. Semantic search and embeddings
5. Local LLM served with vLLM
6. Agent tools and agent creation
7. Topic analysis with KMeans
8. Topic analysis with LDA (phrases of 2 to 3 words)
9. Plots: cluster scatterplot, LDA scatterplot and phrases per topic
10. Limitations and next steps

In [ ]:
"""Install dependencies / Instala as dependências do projeto."""
!pip install -q kagglehub pandas langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface huggingface_hub sentence-transformers
!pip install langchain-nvidia-ai-endpoints
!pip install langchain_community

In [ ]:
import os
import glob
import torch
import kagglehub

import pandas as pd
from google.colab import userdata
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DataFrameLoader

from langchain.agents import create_agent
from langchain_core.tools import create_retriever_tool
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
"""Load secrets and download the dataset / Carrega os secrets e baixa o dataset."""

# hugging face token for the llm endpoint / token do hugging face para o endpoint do llm
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.environ["HF_TOKEN"]

# kaggle token read from colab secrets / token do kaggle lido dos secrets do colab
os.environ["KAGGLE_API_TOKEN"] = userdata.get("kaggle_api")

# download dataset to the local cache / baixa o dataset para o cache local
dataset_path = kagglehub.dataset_download("magodobolo/nvidia-news")

print(dataset_path)
print(os.listdir(dataset_path))

In [ ]:
"""Load the csv and inspect its structure / Carrega o csv e inspeciona a estrutura."""

# find csv files inside the dataset folder / localiza os arquivos csv dentro da pasta do dataset
csv_files = glob.glob(os.path.join(dataset_path, "*.csv"))
print(csv_files)

In [ ]:
# read the first csv file / lê o primeiro arquivo csv
df = pd.read_csv(csv_files[0])

# structure and sample rows / estrutura e amostra das linhas
df.head()

In [ ]:
## GPU
"""Check the gpu before loading the model / Verifica a gpu antes de carregar o modelo."""

# gpu name and total memory in gb / nome da gpu e memória total em gb
props = torch.cuda.get_device_properties(0)
print(props.name)
print(round(props.total_memory / 1e9, 1))

In [ ]:
"""Build documents and the vector store / Cria os documentos e o vector store."""

# columns used as text, adjust after inspecting the data / colunas usadas como texto, ajuste após inspecionar os dados
text_columns = df.select_dtypes(include="object").columns.tolist()

# row limit to keep embedding time low / limite de linhas para manter o tempo de embedding baixo
max_rows = 2000
df_sample = df.head(max_rows)

In [ ]:
# one document per row, joining the text columns / um documento por linha, juntando as colunas de texto
documents = [
    Document(
        page_content="\n".join(
            f"{col}: {row[col]}" for col in text_columns if pd.notna(row[col])
        ),
        metadata={"row": idx},
    )
    for idx, row in df_sample.iterrows()
]

In [ ]:
# split long texts into chunks / divide textos longos em chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documents)

In [ ]:
# embeddings run on gpu when available / os embeddings rodam na gpu quando disponível
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
)

In [ ]:
# index chunks in memory / indexa os chunks em memória
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(len(documents), len(chunks))

In [ ]:
# Model LLM
model_id = "nvidia/Llama-3.1-Nemotron-Nano-8B-v1"

In [ ]:
# tokenizer and model in bf16, the g4 supports it / tokenizer e modelo em bf16, a g4 suporta
tokenizer = AutoTokenizer.from_pretrained(model_id, token=os.environ["HF_TOKEN"])

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    token=os.environ["HF_TOKEN"],
)

In [ ]:
# text generation pipeline, greedy decoding / pipeline de geração de texto, decodificação greedy
gen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
)

In [ ]:
# langchain chat wrapper over the local pipeline / wrapper de chat do langchain sobre o pipeline local
llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipe))

print(round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
## Prompt Agente
# this model toggles reasoning through the system prompt / este modelo liga ou desliga o raciocínio pelo system prompt
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "detailed thinking off\n"
            "You are an assistant specialized in NVIDIA news. "
            "Answer only with the provided context and in the language of the question. "
            "If the context is not enough, say so.",
        ),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ]
)

chain = prompt | llm | StrOutputParser()

In [ ]:
"""Retrieve passages and answer / Recupera os trechos e responde."""

# user question / pergunta do usuário
question = "Quais são as principais novidades recentes da NVIDIA?"

# retrieve relevant chunks / recupera os chunks relevantes
context = "\n\n".join(doc.page_content for doc in retriever.invoke(question))

# generate the answer on the local gpu / gera a resposta na gpu local
answer = chain.invoke({"context": context, "question": question})
print(answer)

In [ ]:
"""Retrieve passages and answer / Recupera os trechos e responde."""

# user question / pergunta do usuário
question = "Explicar sobe o que dilatação temporal"

# retrieve relevant chunks / recupera os chunks relevantes
context = "\n\n".join(doc.page_content for doc in retriever.invoke(question))

# generate the answer on the local gpu / gera a resposta na gpu local
answer = chain.invoke({"context": context, "question": question})
print(answer)

# Agente

In [ ]:
"""Install vllm and the agent dependencies / Instala o vllm e as dependências do agente."""

!pip install -q vllm langchain-openai langgraph requests

In [ ]:
"""Compare memory and disk versions and check the missing class / Compara versões da memória e do disco e verifica a classe ausente."""

from importlib.metadata import version

import langchain_core
import langchain_core.exceptions as core_exceptions

# version loaded in memory vs installed on disk / versão carregada na memória vs instalada no disco
print("memory:", langchain_core.__version__)
print("disk:", version("langchain-core"))

# does the class that langchain-openai needs exist / a classe que o langchain-openai precisa existe
print("ModelAPIError:", hasattr(core_exceptions, "ModelAPIError"))

"""List available core versions and check conflicts / Lista versões disponíveis do core e verifica conflitos."""

!pip index versions langchain-core
!pip check

In [ ]:
"""Find where langchain_core is loaded from and how long the session has been up / Descobre de onde o langchain_core é carregado e há quanto tempo a sessão está ativa."""

import time
from importlib.metadata import distributions

import langchain_core
import psutil

# file actually loaded in memory / arquivo realmente carregado na memória
print("loaded from:", langchain_core.__file__)

# every installed copy of the package / todas as cópias instaladas do pacote
for dist in distributions():
    if dist.metadata["Name"] and dist.metadata["Name"].lower() == "langchain-core":
        print("installed:", dist.version, dist.locate_file(""))

# minutes since this python process started / minutos desde que este processo python iniciou
started = psutil.Process().create_time()
print("session age (min):", round((time.time() - started) / 60, 1))

In [ ]:
"""Force a clean kernel restart / Força um reinício limpo do kernel."""

import os

# kill the python process, colab restarts it automatically / encerra o processo python, o colab o reinicia automaticamente
os.kill(os.getpid(), 9)

In [ ]:
!pip install sentence-transformers

In [ ]:
import subprocess
import time
import requests
from langchain.agents import create_agent
from langchain_core.tools import create_retriever_tool
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

import os
import kagglehub
import glob
import pandas as pd
import subprocess
import time
import requests
import torch
from google.colab import userdata
import sentence_transformers
from transformers import PreTrainedModel

"""Confirm that memory and disk versions match / Confirma que as versões da memória e do disco coincidem."""

from importlib.metadata import version
import langchain_core
import langchain_core.exceptions as core_exceptions
import torch
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

import io
import numpy as np
from langchain.agents import create_agent
from langchain_core.tools import create_retriever_tool, tool
from langchain_openai import ChatOpenAI

# both versions must be equal after the restart / as duas versões devem ser iguais após o reinício
print("memory:", langchain_core.__version__)
print("disk:", version("langchain-core"))
print("ModelAPIError:", hasattr(core_exceptions, "ModelAPIError"))

In [ ]:
"""Check that sentence-transformers imports cleanly / Verifica se o sentence-transformers importa sem erro."""

# versions and gpu visibility / versões e visibilidade da gpu
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("gpu available:", torch.cuda.is_available())

In [ ]:
"""Load secrets and download the kaggle dataset / Carrega os secrets e baixa o dataset do kaggle."""

# hugging face token, also used to download the model / token do hugging face, também usado para baixar o modelo
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.environ["HF_TOKEN"]

# kaggle token read from colab secrets / token do kaggle lido dos secrets do colab
os.environ["KAGGLE_API_TOKEN"] = userdata.get("kaggle_api")

# download the dataset to the local cache / baixa o dataset para o cache local
dataset_path = kagglehub.dataset_download("magodobolo/nvidia-news")

print(dataset_path)
print(os.listdir(dataset_path))

In [ ]:
"""Load the csv and inspect its structure / Carrega o csv e inspeciona a estrutura."""

# find csv files inside the dataset folder / localiza os arquivos csv dentro da pasta do dataset
csv_files = glob.glob(os.path.join(dataset_path, "*.csv"))
print(csv_files)

# read the first csv file / lê o primeiro arquivo csv
df = pd.read_csv(csv_files[0])
df

In [ ]:
# structure and sample rows / estrutura e amostra das linhas
df.info()

In [ ]:
# structure and sample rows / estrutura e amostra das linhas
df.head()

In [ ]:
"""Build documents, embeddings and the retriever / Cria os documentos, os embeddings e o retriever."""

# text columns used for semantic search, adjust after inspecting the data / colunas de texto usadas na busca semântica, ajuste após inspecionar os dados
text_columns = df.select_dtypes(include="object").columns.tolist()

# row limit to keep embedding time low / limite de linhas para manter o tempo de embedding baixo
max_rows = 2000
df_sample = df.head(max_rows)

# one document per row, joining the text columns / um documento por linha, juntando as colunas de texto
documents = [
    Document(
        page_content="\n".join(
            f"{col}: {row[col]}" for col in text_columns if pd.notna(row[col])
        ),
        metadata={"row": idx},
    )
    for idx, row in df_sample.iterrows()
]

In [ ]:
# split long texts into chunks / divide textos longos em chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documents)

# embeddings run on gpu when available / os embeddings rodam na gpu quando disponível
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
)

In [ ]:
# index the chunks in memory / indexa os chunks em memória
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(len(documents), len(chunks))

In [ ]:
"""Start the vllm server in the background / Inicia o servidor vllm em segundo plano."""

model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8"
parser_path = r"/content/nano_v3_reasoning_parser.py"
log_path = r"/content/vllm.log"

# download the reasoning parser plugin from the model repo / baixa o plugin de reasoning parser do repositório do modelo
parser_url = f"https://huggingface.co/{model_id}/resolve/main/nano_v3_reasoning_parser.py"
subprocess.run(["wget", "-q", parser_url, "-O", parser_path], check=True)

# server command with tool calling enabled / comando do servidor com tool calling habilitado
vllm_cmd = [
    "vllm", "serve", model_id,
    "--served-model-name", "model",
    "--port", "8000",
    "--max-model-len", "32768",
    "--gpu-memory-utilization", "0.80",
    "--trust-remote-code",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--reasoning-parser-plugin", parser_path,
    "--reasoning-parser", "nano_v3",
]

In [ ]:
# log to file to avoid flooding the notebook / grava em arquivo para não poluir o notebook
log_file = open(log_path, "w")
server = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

# wait until the server answers, first run downloads the weights / espera o servidor responder, a primeira execução baixa os pesos
ready = False
for _ in range(180):
    if server.poll() is not None:
        break
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).status_code == 200:
            ready = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(10)

print("server ready" if ready else "server failed, check /content/vllm.log")

In [ ]:
"""Check nvcc and the cuda arch settings flashinfer relies on / Verifica o nvcc e as configurações de arquitetura cuda que o flashinfer usa."""

import os
import shutil
import subprocess

import torch

# gpu compute capability, this card should report 12.0 / capacidade de computação da gpu, esta placa deve informar 12.0
print("capability:", torch.cuda.get_device_capability(0))

# nvcc is needed because flashinfer compiles kernels on the fly / o nvcc é necessário porque o flashinfer compila kernels na hora
nvcc_path = shutil.which("nvcc")
print("nvcc path:", nvcc_path)
if nvcc_path:
    print(subprocess.run([nvcc_path, "--version"], capture_output=True, text=True).stdout)

# environment variables that influence the target architectures / variáveis de ambiente que influenciam as arquiteturas alvo
for name in ["CUDA_HOME", "TORCH_CUDA_ARCH_LIST", "FLASHINFER_CUDA_ARCH_LIST"]:
    print(name, "=", os.environ.get(name))

In [ ]:
"""Start the vllm server with an explicit flashinfer target architecture / Inicia o servidor vllm com a arquitetura alvo do flashinfer definida explicitamente."""

import os
import shutil
import subprocess
import time

import requests

model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8"
parser_path = r"/content/nano_v3_reasoning_parser.py"
log_path = r"/content/vllm.log"

# stop leftovers from previous attempts / encerra restos de tentativas anteriores
subprocess.run(["pkill", "-f", "vllm serve"])

# download the reasoning parser plugin from the model repo / baixa o plugin de reasoning parser do repositório do modelo
parser_url = f"https://huggingface.co/{model_id}/resolve/main/nano_v3_reasoning_parser.py"
subprocess.run(["wget", "-q", parser_url, "-O", parser_path], check=True)

# environment for the server process / ambiente do processo do servidor
server_env = {**os.environ, "FLASHINFER_CUDA_ARCH_LIST": "12.0a"}

# point cuda home to the toolkit that holds nvcc when it is not set / aponta o cuda home para o toolkit que tem o nvcc quando não está definido
nvcc_path = shutil.which("nvcc")
if nvcc_path and "CUDA_HOME" not in server_env:
    server_env["CUDA_HOME"] = os.path.dirname(os.path.dirname(nvcc_path))

# server command with tool calling enabled / comando do servidor com tool calling habilitado
vllm_cmd = [
    "vllm", "serve", model_id,
    "--served-model-name", "model",
    "--port", "8000",
    "--max-model-len", "32768",
    "--gpu-memory-utilization", "0.80",
    "--trust-remote-code",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--reasoning-parser-plugin", parser_path,
    "--reasoning-parser", "nano_v3",
]

# log to file to avoid flooding the notebook / grava em arquivo para não poluir o notebook
log_file = open(log_path, "w")
server = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT, env=server_env)

# wait until the server answers, kernel compilation can add minutes / espera o servidor responder, a compilação de kernels pode somar minutos
ready = False
for _ in range(180):
    if server.poll() is not None:
        break
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).status_code == 200:
            ready = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(10)

print("server ready" if ready else "server failed, check /content/vllm.log")

In [ ]:
# llm client pointing to the local vllm server / cliente do llm apontando para o servidor vllm local
llm = ChatOpenAI(
    model="model",
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    temperature=0.1,
    max_tokens=4096,
)

# semantic search over the news texts / busca semântica nos textos das notícias
retriever_tool = create_retriever_tool(
    retriever,
    name="nvidia_news_search",
    description=(
        "Search the NVIDIA news dataset and return relevant passages / "
        "Busca no dataset de notícias da NVIDIA e retorna trechos relevantes."
    ),
)

# analysis tools plus the semantic search / ferramentas de análise mais a busca semântica
tools = [dataset_overview, run_pandas, retriever_tool]

In [ ]:
# agent that plans, runs code and explains the result / agente que planeja, executa código e explica o resultado
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a data analyst working on the NVIDIA news dataset, loaded as the pandas dataframe `df`. "
        "Always call dataset_overview first to learn the columns. "
        "Use run_pandas for any count, filter, group, trend or statistic, assign the final value to `result`, "
        "and never invent numbers. "
        "Use nvidia_news_search only for questions about the meaning of the news texts. "
        "Use only passages directly about NVIDIA and ignore unrelated ones. "
        "If the code fails, read the error and try again. "
        "Answer in the language of the question and say which columns you used."
    ),
)

In [ ]:
"""Ask the agent and show each step / Faz a pergunta ao agente e mostra cada passo."""

# analysis question, adjust to your columns / pergunta de análise, ajuste para as suas colunas
question = "Faça uma análise exploratória do dataset: tamanho, colunas, valores nulos e os principais padrões."

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"recursion_limit": 30},
)

# print tool calls, tool results and the final answer / imprime chamadas de ferramenta, resultados e a resposta final
for message in response["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print("tool call:", call["name"], call["args"])
    elif message.type == "tool":
        print("tool result:", message.content[:500])
    elif message.type == "ai":
        print("answer:", message.content)

In [ ]:
"""Run several questions in a row / Executa várias perguntas em sequência."""

# list of questions, edit freely / lista de perguntas, edite à vontade
questions = [
    "Quantas notícias existem no dataset e quais são as colunas?",
    "Quais são os temas mais frequentes nas notícias sobre a NVIDIA?",
    "Existe alguma tendência ao longo do tempo no volume de notícias?",
]

for question in questions:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 30},
    )
    print("question:", question)
    print("answer:", response["messages"][-1].content)
    print()

# **Agente com analise de tópicos**

In [ ]:
import json
import os
import shutil
import subprocess
import time
import requests
import matplotlib.pyplot as plt
from langchain_core.tools import tool
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
"""Reuse the vllm server if alive, otherwise restart it and show the log on failure / Reaproveita o servidor vllm se estiver vivo, senão reinicia e mostra o log em caso de falha."""

model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8"
parser_path = r"/content/nano_v3_reasoning_parser.py"
log_path = r"/content/vllm.log"

In [ ]:
"""Embed headlines and summaries for topic analysis / Gera embeddings de manchetes e resumos para a análise de tópicos."""

# multilingual model, the texts are mostly in portuguese / modelo multilíngue, os textos estão majoritariamente em português
topic_encoder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# one text per row: headline plus a slice of the summary / um texto por linha: manchete mais um trecho do resumo
topic_texts = (df["Headline"].astype(str) + ". " + df["Summary"].astype(str).str.slice(0, 500)).tolist()

# normalized embeddings make kmeans behave like cosine clustering / embeddings normalizados fazem o kmeans se comportar como clustering por cosseno
topic_embeddings = topic_encoder.encode(
    topic_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
print(topic_embeddings.shape)

In [ ]:
# stop words in portuguese and english plus company names / stop words em português e inglês mais nomes de empresas
pt_stop_words = [
    "de", "a", "o", "que", "e", "do", "da", "em", "um", "para", "com", "não", "uma", "os", "no", "se", "na",
    "por", "mais", "as", "dos", "como", "mas", "ao", "das", "seu", "sua", "ou", "quando", "muito", "nos",
    "já", "também", "só", "pelo", "pela", "até", "isso", "entre", "depois", "sem", "mesmo", "aos", "seus",
    "nas", "esse", "essa", "é", "são", "foi", "será", "ser", "tem", "ter", "pode", "sobre", "após", "novo",
    "nova", "segundo", "disse", "diz", "ainda", "nvidia", "spotify",
]
topic_stop_words = list(ENGLISH_STOP_WORDS) + pt_stop_words

In [ ]:
# shared state between the two tools / estado compartilhado entre as duas ferramentas
topic_state = {}


@tool
def cluster_topics(n_clusters: int = 6, company: str = "") -> str:
    """Group news into topic clusters and return size, keywords and sample headlines of each cluster, optionally filtered by company / Agrupa notícias em clusters de tópicos e retorna tamanho, palavras-chave e manchetes de exemplo de cada cluster, com filtro opcional por empresa."""
    # rows to cluster, all or a single company / linhas a agrupar, todas ou de uma única empresa
    if company:
        mask = df["Company"].str.upper() == company.upper()
    else:
        mask = df["Company"].notna()
    index = np.where(mask.to_numpy())[0]
    if len(index) < n_clusters * 3:
        return "error: too few rows for this number of clusters / poucas linhas para este número de clusters"

    # kmeans on the normalized embeddings / kmeans nos embeddings normalizados
    vectors = topic_embeddings[index]
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(vectors)
    labels = kmeans.labels_
    distances = kmeans.transform(vectors)

    topic_state["index"] = index
    topic_state["labels"] = labels
    topic_state["company"] = company or "all"

    # one document per cluster to extract distinctive terms with tf-idf / um documento por cluster para extrair termos distintivos com tf-idf
    cluster_docs = [" ".join(topic_texts[i] for i in index[labels == c]) for c in range(n_clusters)]
    vectorizer = TfidfVectorizer(stop_words=topic_stop_words, max_df=0.6, min_df=1)
    tfidf = vectorizer.fit_transform(cluster_docs)
    terms = np.array(vectorizer.get_feature_names_out())

    silhouette = silhouette_score(vectors, labels)
    lines = [f"rows: {len(index)}, clusters: {n_clusters}, silhouette: {silhouette:.3f}"]

    for c in range(n_clusters):
        top_terms = terms[np.argsort(tfidf[c].toarray()[0])[::-1][:8]]
        members = np.where(labels == c)[0]

        # headlines closest to the cluster center / manchetes mais próximas do centro do cluster
        closest = members[np.argsort(distances[members, c])[:3]]
        samples = [df["Headline"].iloc[index[m]] for m in closest]

        lines.append(f"cluster {c} | size {len(members)} | keywords: {', '.join(top_terms)}")
        for sample_headline in samples:
            lines.append(f"  - {sample_headline}")

    return "\n".join(lines)


@tool
def plot_clusters(cluster_names: str = "{}") -> str:
    """Plot the 2D map of the clusters from the last cluster_topics call, cluster_names is a JSON object like {"0": "name"} / Plota o mapa 2D dos clusters da última chamada de cluster_topics, cluster_names é um objeto JSON como {"0": "nome"}."""
    if "labels" not in topic_state:
        return "error: call cluster_topics first / chame cluster_topics primeiro"

    # names given by the agent, fall back to the cluster number / nomes dados pelo agente, usa o número do cluster como alternativa
    try:
        names = {int(k): str(v) for k, v in json.loads(cluster_names).items()}
    except (json.JSONDecodeError, ValueError, AttributeError):
        names = {}

    index = topic_state["index"]
    labels = topic_state["labels"]

    # project the embeddings to 2D for the map / projeta os embeddings para 2D para o mapa
    perplexity = min(30, max(5, len(index) // 10))
    coords = TSNE(n_components=2, perplexity=perplexity, init="pca", random_state=42).fit_transform(
        topic_embeddings[index]
    )

    fig, ax = plt.subplots(figsize=(11, 7))
    colors = plt.cm.tab10.colors
    for c in sorted(set(labels)):
        points = coords[labels == c]
        label = f"{c}: {names.get(c, 'cluster ' + str(c))} ({len(points)})"
        ax.scatter(points[:, 0], points[:, 1], s=18, alpha=0.75, color=colors[c % len(colors)], label=label)

    ax.set_title(f"Topic clusters / Clusters de tópicos ({topic_state['company']})")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
    fig.tight_layout()

    # save and show inline in the notebook / salva e mostra inline no notebook
    fig.savefig(r"/content/topic_clusters.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return "plot shown and saved to /content/topic_clusters.png / gráfico exibido e salvo em /content/topic_clusters.png"

In [ ]:
# check whether the server already answers / verifica se o servidor já responde
try:
    alive = requests.get("http://localhost:8000/v1/models", timeout=3).status_code == 200
except requests.exceptions.RequestException:
    alive = False

print("server alive:", alive)

if not alive:
    # save the previous log before it is overwritten / guarda o log anterior antes de ser sobrescrito
    if os.path.exists(log_path):
        shutil.copy(log_path, r"/content/vllm_previous.log")

    # stop leftovers and show gpu memory / encerra restos e mostra a memória da gpu
    subprocess.run(["pkill", "-f", "vllm serve"])
    time.sleep(5)
    print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

    # reasoning parser plugin / plugin de reasoning parser
    parser_url = f"https://huggingface.co/{model_id}/resolve/main/nano_v3_reasoning_parser.py"
    subprocess.run(["wget", "-q", parser_url, "-O", parser_path], check=True)

    # explicit flashinfer architecture, needed on this blackwell gpu / arquitetura do flashinfer explícita, necessária nesta gpu blackwell
    server_env = {**os.environ, "FLASHINFER_CUDA_ARCH_LIST": "12.0a"}
    nvcc_path = shutil.which("nvcc")
    if nvcc_path and "CUDA_HOME" not in server_env:
        server_env["CUDA_HOME"] = os.path.dirname(os.path.dirname(nvcc_path))

    vllm_cmd = [
        "vllm", "serve", model_id,
        "--served-model-name", "model",
        "--port", "8000",
        "--max-model-len", "32768",
        "--gpu-memory-utilization", "0.70",
        "--trust-remote-code",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--reasoning-parser-plugin", parser_path,
        "--reasoning-parser", "nano_v3",
    ]

    log_file = open(log_path, "w")
    server = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT, env=server_env)

    # wait for the server, startup can take several minutes / espera o servidor, a inicialização pode levar vários minutos
    for _ in range(180):
        if server.poll() is not None:
            break
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=2).status_code == 200:
                alive = True
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(10)

    print("server ready" if alive else "server failed")

    # when it fails, show the end of the log / quando falha, mostra o final do log
    if not alive:
        with open(log_path) as log_file:
            print("".join(log_file.readlines()[-40:]))

In [ ]:
"""Recreate the agent with the topic tools / Recria o agente com as ferramentas de tópicos."""

from langchain.agents import create_agent

# previous tools plus the two new ones / ferramentas anteriores mais as duas novas
tools = [dataset_overview, run_pandas, retriever_tool, cluster_topics, plot_clusters]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a data analyst working on a news dataset, loaded as the pandas dataframe `df`. "
        "Use run_pandas for counts and statistics and never invent numbers. "
        "For topic analysis: first call cluster_topics, using company='NVIDIA' when the question is about NVIDIA. "
        "Then give each cluster a short name in Portuguese based on its keywords and sample headlines, "
        "and call plot_clusters with a JSON object that maps each cluster number to its name. "
        "Finally explain each topic in one sentence, with its size and one example headline. "
        "Only describe what the tool results show. Answer in the language of the question."
    ),
)

In [ ]:
"""Ask the agent for the topic analysis with the cluster map / Pede ao agente a análise de tópicos com o mapa de clusters."""

question = "Faça uma análise de tópicos das notícias da NVIDIA em 6 clusters, nomeie cada tópico e mostre o gráfico dos clusters."

response = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"recursion_limit": 30},
)

# the plot appears while the agent runs, the steps are printed after / o gráfico aparece enquanto o agente roda, os passos são impressos depois
for message in response["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print("tool call:", call["name"], call["args"])
    elif message.type == "tool":
        print("tool result:", message.content[:1500])
    elif message.type == "ai":
        print("answer:", message.content)

In [ ]:
"""Find where the journalist names appear in the text / Encontra onde os nomes dos jornalistas aparecem no texto."""

# rows that contain the suspicious tokens / linhas que contêm os termos suspeitos
mask = df["Summary"].str.contains("Nellis|Chauhan|Shristi", case=False, regex=True)
print("rows:", mask.sum())

# end of the summaries, where bylines usually live / final dos resumos, onde as assinaturas costumam ficar
for text in df.loc[mask, "Summary"].head(5):
    print(repr(text[-200:]))

In [ ]:
"""Extend the stop words with bylines and rerun the tool check / Estende as stop words com assinaturas e refaz a verificação da ferramenta."""

# journalist names and boilerplate seen in the keywords / nomes de jornalistas e texto padrão vistos nas palavras-chave
topic_stop_words.extend(["stephen", "nellis", "shashwat", "chauhan", "shristi", "pro", "reuters"])

# run the tool directly to read the full result without truncation / chama a ferramenta direto para ler o resultado completo sem corte
print(cluster_topics.invoke({"n_clusters": 6, "company": "NVIDIA"}))

In [ ]:
"""Compare the silhouette score for several cluster counts / Compara o silhouette para vários números de clusters."""

# nvidia rows only, same filter as the agent / apenas linhas da nvidia, mesmo filtro do agente
index = np.where((df["Company"].str.upper() == "NVIDIA").to_numpy())[0]
vectors = topic_embeddings[index]

for k in range(3, 13):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(vectors)
    print(k, round(silhouette_score(vectors, labels), 3))

In [ ]:
"""Strip news agency bylines and rebuild the embeddings / Remove as assinaturas da agência de notícias e refaz os embeddings."""

import re

# remove the reporting credit in parentheses / remove o crédito de reportagem entre parênteses
byline_pattern = re.compile(r"\((?:Reportagem|Reporting|Edição)[^)]*\)", flags=re.IGNORECASE)

clean_summary = df["Summary"].astype(str).str.replace(byline_pattern, "", regex=True).str.strip()
print("rows changed:", (clean_summary != df["Summary"].astype(str)).sum())

# rebuild the texts and embeddings used by cluster_topics / refaz os textos e embeddings usados pelo cluster_topics
topic_texts = (df["Headline"].astype(str) + ". " + clean_summary.str.slice(0, 500)).tolist()
topic_embeddings = topic_encoder.encode(
    topic_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)

# run the tool directly to read the full result / chama a ferramenta direto para ler o resultado completo
print(cluster_topics.invoke({"n_clusters": 8, "company": "NVIDIA"}))

In [ ]:
"""Check how many nvidia-labeled rows never mention nvidia / Verifica quantas linhas rotuladas como nvidia nunca mencionam a nvidia."""

# rows labeled as NVIDIA / linhas rotuladas como NVIDIA
is_nvidia = df["Company"].str.upper() == "NVIDIA"

# does the headline or the summary mention the company / a manchete ou o resumo menciona a empresa
text_all = df["Headline"].astype(str) + " " + df["Summary"].astype(str)
mentions = text_all.str.contains("nvidia", case=False)

print("nvidia rows:", is_nvidia.sum())
print("nvidia rows without the word nvidia:", (is_nvidia & ~mentions).sum())

# a few examples / alguns exemplos
display(df.loc[is_nvidia & ~mentions, ["Headline", "Date"]].head(10))

In [ ]:
"""Show full headlines of nvidia rows without the word nvidia and look for other mentions / Mostra manchetes completas das linhas sem a palavra nvidia e procura outras menções."""

# rows labeled as NVIDIA that never use the word / linhas rotuladas como NVIDIA que nunca usam a palavra
subset = df.loc[is_nvidia & ~mentions].copy()

# other ways the company can be mentioned / outras formas de citar a empresa
subset["has_ticker_or_ceo"] = text_all.loc[subset.index].str.contains("NVDA|Huang|Jensen", case=False)
print("rows citing ticker or ceo:", subset["has_ticker_or_ceo"].sum(), "of", len(subset))

# full headlines without truncation / manchetes completas sem truncamento
with pd.option_context("display.max_colwidth", None):
    display(subset[["Headline", "Date", "has_ticker_or_ceo"]])

In [ ]:
"""Share of rows that name nvidia in the headline, per topic cluster / Proporção de linhas que citam a nvidia na manchete, por cluster de tópico."""

# rows and labels from the last cluster_topics call / linhas e rótulos da última chamada do cluster_topics
rows = df.iloc[topic_state["index"]].copy()
rows["cluster"] = topic_state["labels"]

# does the headline itself name the company / a manchete cita a empresa
rows["nvidia_in_headline"] = rows["Headline"].astype(str).str.contains("nvidia|nvda", case=False)

# per cluster: size and share naming nvidia in the headline / por cluster: tamanho e proporção que cita a nvidia na manchete
summary = rows.groupby("cluster")["nvidia_in_headline"].agg(size="size", share="mean").round(2)
display(summary)

# Agente LDA Extração tipos

In [ ]:
import re
import json
import matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from langchain_core.tools import tool
from sklearn.manifold import TSNE
from langchain.agents import create_agent
from IPython.display import Image, display
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
"""Prepare clean texts and deduplicated rows for LDA / Prepara textos limpos e linhas sem duplicatas para o LDA."""
# news agency credit lines in parentheses / linhas de crédito da agência de notícias entre parênteses
byline_pattern = re.compile(r"\((?:Reportagem|Reporting|Edição)[^)]*\)", flags=re.IGNORECASE)
clean_summary = df["Summary"].astype(str).str.replace(byline_pattern, "", regex=True)

# headline plus summary, without links and numbers / manchete mais resumo, sem links e números
lda_texts = (
    (df["Headline"].astype(str) + ". " + clean_summary)
    .str.replace(r"https?://\S+", " ", regex=True)
    .str.replace(r"\d+(?:[.,]\d+)*", " ", regex=True)
    .tolist()
)

# keep one row per normalized headline, repeated daily reports collapse into one / mantém uma linha por manchete normalizada, relatórios diários repetidos viram um só
headline_norm = (
    df["Headline"].astype(str).str.lower().str.replace(r"\d+(?:[.,]\d+)*", "", regex=True).str.strip()
)
lda_keep = (~headline_norm.duplicated()).to_numpy()
print("rows kept:", int(lda_keep.sum()), "of", len(df))

In [ ]:
# stop words from the clustering step plus agency boilerplate / stop words do clustering mais texto padrão da agência
lda_stop_words = list(topic_stop_words) + ["reportagem", "adicional", "edição", "reuters", "stephen", "nellis"]

In [ ]:
"""Create the LDA phrase topic tool and its plot tool / Cria a ferramenta de tópicos por frases com LDA e a ferramenta de gráfico."""
# shared state between the two tools / estado compartilhado entre as duas ferramentas
lda_state = {}

@tool
def lda_topics(n_topics: int = 6, company: str = "", ngram_min: int = 2, ngram_max: int = 3) -> str:
    """Fit an LDA topic model on the news using word phrases and return each topic with its top phrases, size and representative headlines / Ajusta um modelo LDA nas notícias usando frases e retorna cada tópico com suas principais frases, tamanho e manchetes representativas."""
    # rows to model: optional company filter and deduplicated headlines / linhas a modelar: filtro opcional de empresa e manchetes sem duplicatas
    if company:
        company_mask = (df["Company"].str.upper() == company.upper()).to_numpy()
    else:
        company_mask = np.ones(len(df), dtype=bool)
    index = np.where(company_mask & lda_keep)[0]
    docs = [lda_texts[i] for i in index]

    # phrase counts, very rare and very common phrases are dropped / contagem de frases, frases muito raras e muito comuns são descartadas
    vectorizer = CountVectorizer(
        stop_words=lda_stop_words,
        ngram_range=(ngram_min, ngram_max),
        min_df=5,
        max_df=0.5,
        token_pattern=r"(?u)\b[^\W\d_]{3,}\b",
    )
    counts = vectorizer.fit_transform(docs)
    terms = np.array(vectorizer.get_feature_names_out())
    if counts.shape[1] < n_topics * 5:
        return "error: too few phrases, lower ngram_min or n_topics / poucas frases, reduza ngram_min ou n_topics"

    # low priors favor few topics per text and few phrases per topic / priors baixos favorecem poucos tópicos por texto e poucas frases por tópico
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        doc_topic_prior=0.1,
        topic_word_prior=0.01,
        learning_method="batch",
        max_iter=40,
        random_state=42,
    )
    doc_topic = lda.fit_transform(counts)
    components = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
    labels = doc_topic.argmax(axis=1)
    strength = doc_topic.max(axis=1)

    # umass coherence on the top phrases of each topic / coerência umass nas principais frases de cada tópico
    binary = (counts > 0).astype(np.int32).tocsc()
    coherences = []
    for t in range(n_topics):
        top_idx = np.argsort(components[t])[::-1][:10]
        sub = binary[:, top_idx].toarray()
        cooc = sub.T @ sub
        doc_freq = np.diag(cooc)
        score = 0.0
        for m in range(1, len(top_idx)):
            for l in range(m):
                score += np.log((cooc[m, l] + 1) / doc_freq[l])
        coherences.append(score / (len(top_idx) * (len(top_idx) - 1) / 2))

    # state shared with the plot tools / estado compartilhado com as ferramentas de gráfico
    lda_state.update(
        {
            "components": components,
            "terms": terms,
            "sizes": np.bincount(labels, minlength=n_topics),
            "company": company or "all",
            "coherence": float(np.mean(coherences)),
            "perplexity": float(lda.perplexity(counts)),
            "share_strong": float((strength >= 0.5).mean()),
            "doc_topic": doc_topic,
            "index": index,
            "labels": labels,
            "strength": strength,
        }
    )

    lines = [
        f"rows: {len(index)}, phrases: {counts.shape[1]}, topics: {n_topics}, "
        f"mean coherence (umass): {lda_state['coherence']:.2f}, "
        f"share of texts with a dominant topic above 0.5: {lda_state['share_strong']:.2f}"
    ]
    for t in range(n_topics):
        top_phrases = terms[np.argsort(components[t])[::-1][:8]]

        # headlines where this topic weighs the most / manchetes onde este tópico pesa mais
        best = np.argsort(doc_topic[:, t])[::-1][:3]
        samples = [f"{df['Headline'].iloc[index[b]]} (weight {doc_topic[b, t]:.2f})" for b in best]

        lines.append(
            f"topic {t} | texts {int((labels == t).sum())} | coherence {coherences[t]:.2f} | phrases: {', '.join(top_phrases)}"
        )
        lines.extend(f"  - {sample}" for sample in samples)

    return "\n".join(lines)

In [ ]:
@tool
def plot_lda_topics(topic_names: str = "{}") -> str:
    """Plot the top phrases of each topic from the last lda_topics call, topic_names is a JSON object like {"0": "name"} / Plota as principais frases de cada tópico da última chamada de lda_topics, topic_names é um objeto JSON como {"0": "nome"}."""
    if "components" not in lda_state:
        return "error: call lda_topics first / chame lda_topics primeiro"

    # names given by the agent, fall back to the topic number / nomes dados pelo agente, usa o número do tópico como alternativa
    try:
        names = {int(k): str(v) for k, v in json.loads(topic_names).items()}
    except (json.JSONDecodeError, ValueError, AttributeError):
        names = {}

    components = lda_state["components"]
    terms = lda_state["terms"]
    n_topics = components.shape[0]
    n_cols = 2
    n_rows = int(np.ceil(n_topics / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 3.2 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    colors = plt.cm.tab10.colors

    for t in range(n_topics):
        top_idx = np.argsort(components[t])[::-1][:8]
        axes[t].barh(terms[top_idx][::-1], components[t][top_idx][::-1], color=colors[t % len(colors)])
        axes[t].set_title(f"{t}: {names.get(t, 'topic ' + str(t))} ({int(lda_state['sizes'][t])} textos)", fontsize=10)
        axes[t].tick_params(labelsize=8)

    # hide unused panels / esconde painéis não usados
    for extra in axes[n_topics:]:
        extra.axis("off")

    fig.suptitle(f"LDA topics by phrases / Tópicos LDA por frases ({lda_state['company']})")
    fig.tight_layout()

    fig.savefig(r"/content/lda_topics.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return "plot shown and saved to /content/lda_topics.png / gráfico exibido e salvo em /content/lda_topics.png"

In [ ]:
@tool
def plot_lda_scatter(topic_names: str = "{}") -> str:
    """Plot a 2D scatterplot of the news colored by dominant LDA topic from the last lda_topics call, topic_names is a JSON object like {"0": "name"} / Plota um scatterplot 2D das notícias coloridas pelo tópico LDA dominante da última chamada de lda_topics, topic_names é um objeto JSON como {"0": "nome"}."""
    if "doc_topic" not in lda_state:
        return "error: call lda_topics first / chame lda_topics primeiro"

    # names given by the agent, fall back to the topic number / nomes dados pelo agente, usa o número do tópico como alternativa
    try:
        names = {int(k): str(v) for k, v in json.loads(topic_names).items()}
    except (json.JSONDecodeError, ValueError, AttributeError):
        names = {}

    doc_topic = lda_state["doc_topic"]
    labels = lda_state["labels"]
    strength = lda_state["strength"]
    n_topics = doc_topic.shape[1]

    # square root of the probabilities makes euclidean distance behave like hellinger distance / a raiz das probabilidades faz a distância euclidiana se comportar como a distância de hellinger
    perplexity = min(30, max(5, len(doc_topic) // 10))
    coords = TSNE(n_components=2, perplexity=perplexity, init="pca", random_state=42).fit_transform(
        np.sqrt(doc_topic)
    )

    fig, ax = plt.subplots(figsize=(11, 7))
    colors = plt.cm.tab10.colors
    for t in range(n_topics):
        mask = labels == t
        label = f"{t}: {names.get(t, 'topic ' + str(t))} ({int(mask.sum())})"

        # point size grows with how dominant the topic is in the text / o tamanho do ponto cresce com o quanto o tópico domina o texto
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            s=10 + 45 * strength[mask], alpha=0.7, color=colors[t % len(colors)], label=label,
        )

        # topic number at the center of its points / número do tópico no centro dos seus pontos
        if mask.any():
            center = np.median(coords[mask], axis=0)
            ax.annotate(str(t), center, fontsize=13, fontweight="bold", ha="center", va="center")

    ax.set_title(f"LDA topics scatterplot / Scatterplot dos tópicos LDA ({lda_state['company']})")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
    fig.tight_layout()

    # save and show inline in the notebook / salva e mostra inline no notebook
    fig.savefig(r"/content/lda_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return "scatterplot shown and saved to /content/lda_scatter.png / scatterplot exibido e salvo em /content/lda_scatter.png"

In [ ]:
"""Recreate the agent with the LDA scatterplot tool / Recria o agente com a ferramenta de scatterplot do LDA."""

# all previous tools plus the new scatterplot / todas as ferramentas anteriores mais o novo scatterplot
tools = [
    dataset_overview, run_pandas, retriever_tool,
    cluster_topics, plot_clusters,
    lda_topics, plot_lda_topics, plot_lda_scatter,
]

In [ ]:
# Prompt eng
question = "Faça uma análise de tópicos com LDA das notícias da NVIDIA usando frases em 6 tópicos, nomeie cada tópico e mostre o gráfico."

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a data analyst working on a news dataset, loaded as the pandas dataframe `df`. "
        "Use run_pandas for counts and statistics and never invent numbers. "
        "When the user asks for LDA or topic phrases, call lda_topics, using company='NVIDIA' for NVIDIA questions. "
        "Name each topic in Portuguese from its representative headlines first and its phrases second. "
        "Then call plot_lda_scatter and plot_lda_topics, both with a JSON object that maps each topic number to the same name. "
        "plot_lda_scatter shows a 2D scatterplot of the news by dominant topic, "
        "plot_lda_topics shows bar charts with the top phrases of each topic. "
        "Always call plot_lda_scatter when the user asks for the scatterplot or the 2D map. "
        "If the phrases of a topic look like advertising, ticker labels, weekdays or template text, say so instead of naming it as a theme. "
        "Copy coherence values and counts exactly from the tool result. "
        "Do not call coherence a quality ranking, because template text inflates it. "
        "Answer in the language of the question."
    ),
)

In [ ]:
"""Ask the agent for the LDA topics with the scatterplot / Pede ao agente os tópicos LDA com o scatterplot."""

question = (
    "Faça uma análise de tópicos com LDA das notícias da NVIDIA usando frases em 8 tópicos, "
    "nomeie cada tópico e mostre o scatterplot dos tópicos e o gráfico de barras das frases."
)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"recursion_limit": 30},
)

# print the full tool results so nothing is cut / imprime os resultados completos das ferramentas para nada ser cortado
for message in response["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print("tool call:", call["name"], call["args"])
    elif message.type == "tool":
        print("tool result:", message.content)
    elif message.type == "ai":
        print("answer:", message.content)

In [ ]:
# backend must be the inline one in colab / o backend deve ser o inline no colab
print("backend:", matplotlib.get_backend())

In [ ]:
# tools the current agent was built with / ferramentas com que o agente atual foi criado
print("agent tools:", [t.name for t in tools])

In [ ]:
# state needed by the plot tools / estado necessário para as ferramentas de gráfico
print("cluster state ready:", "labels" in topic_state)
print("lda state ready:", "doc_topic" in lda_state)

In [ ]:
# kmeans clusters, direct call / clusters do kmeans, chamada direta
print(cluster_topics.invoke({"n_clusters": 6, "company": "NVIDIA"})[:300])
print(plot_clusters.invoke({"cluster_names": "{}"}))

In [ ]:
# lda scatterplot, direct call / scatterplot do lda, chamada direta
print(lda_topics.invoke({"n_topics": 8, "company": "NVIDIA"})[:300])
print(plot_lda_scatter.invoke({"topic_names": "{}"}))

In [ ]:
# show the saved files even if inline display failed / mostra os arquivos salvos mesmo se a exibição inline falhou
for path in [r"/content/topic_clusters.png", r"/content/lda_scatter.png", r"/content/lda_topics.png"]:
    if os.path.exists(path):
        print(path, os.path.getsize(path), "bytes")
        display(Image(filename=path))
    else:
        print("missing:", path)

In [ ]:
"""Find sentences repeated across many news items / Encontra frases repetidas em muitas notícias."""

import re
from collections import Counter

# split each summary into sentences / divide cada resumo em frases
sentence_counts = Counter()
for text in df["Summary"].astype(str):
    sentences = re.split(r"(?<=[.!?])\s+", text)

    # numbers are masked so templates with changing values match / números são mascarados para que modelos com valores variáveis coincidam
    masked = {re.sub(r"\d+(?:[.,]\d+)*", "#", s.strip().lower()) for s in sentences if len(s.strip()) > 20}
    sentence_counts.update(masked)

# most repeated sentences, candidates for boilerplate / frases mais repetidas, candidatas a texto padrão
for sentence, count in sentence_counts.most_common(15):
    print(count, "|", sentence[:150])

In [ ]:
"""Remove repeated sentences and tickers, then rebuild the LDA inputs / Remove frases repetidas e tickers e refaz as entradas do LDA."""

# a sentence found in at least this many news items is treated as boilerplate / uma frase encontrada em pelo menos esta quantidade de notícias é tratada como texto padrão
min_repeats = 8
boilerplate = {s for s, c in sentence_counts.items() if c >= min_repeats}
print("boilerplate sentences:", len(boilerplate))

# exchange labels such as (NASDAQ:MSFT) / rótulos de bolsa como (NASDAQ:MSFT)
ticker_pattern = re.compile(r"\((?:NASDAQ|NYSE|SA|BVMF)[^)]*\)", flags=re.IGNORECASE)

cleaned = []
for text in df["Summary"].astype(str):
    text = byline_pattern.sub("", text)
    text = ticker_pattern.sub("", text)
    sentences = re.split(r"(?<=[.!?])\s+", text)

    # keep only sentences that are not boilerplate / mantém apenas frases que não são texto padrão
    kept = [
        s for s in sentences
        if re.sub(r"\d+(?:[.,]\d+)*", "#", s.strip().lower()) not in boilerplate
    ]
    cleaned.append(" ".join(kept))

# rebuild the texts read by lda_topics, without links and numbers / refaz os textos lidos pelo lda_topics, sem links e números
lda_texts = [
    re.sub(r"\d+(?:[.,]\d+)*", " ", re.sub(r"https?://\S+", " ", headline + ". " + summary))
    for headline, summary in zip(df["Headline"].astype(str), cleaned)
]

# weekdays and exchange labels add noise to the phrases / dias da semana e rótulos de bolsa adicionam ruído às frases
lda_stop_words = list(set(lda_stop_words + [
    "feira", "quinta", "sexta", "quarta", "terça", "segunda", "nasdaq", "nyse", "corporation",
]))

# run the tool directly to read the full result / chama a ferramenta direto para ler o resultado completo
print(lda_topics.invoke({"n_topics": 6, "company": "NVIDIA"}))

In [ ]:
"""Flag template news so topics reflect editorial content / Marca notícias padronizadas para que os tópicos reflitam conteúdo editorial."""

# daily market wrap headlines follow a fixed pattern / manchetes de fechamento diário seguem um padrão fixo
is_daily_wrap = df["Headline"].astype(str).str.contains(
    r"^E\.U\.A\. - Ações fecharam|^Ações - Wall Street|^Ações - Mercado futuro", case=False, regex=True
)

# promotional text that survived the sentence filter / texto promocional que sobreviveu ao filtro de frases
promo_words = r"cupom|investingpro|protips"
is_promo = df["Summary"].astype(str).str.contains(promo_words, case=False, regex=True)

print("daily wraps:", int(is_daily_wrap.sum()))
print("promotional:", int(is_promo.sum()))
print("both flags:", int((is_daily_wrap & is_promo).sum()))

# keep both flags for later comparisons and drop them from the lda input / guarda as duas marcas para comparações e as remove da entrada do lda
lda_keep = lda_keep & (~is_daily_wrap.to_numpy())
print("rows kept for lda:", int((lda_keep & (df["Company"].str.upper() == "NVIDIA").to_numpy()).sum()))

# promotional words are also removed as vocabulary / palavras promocionais também são removidas do vocabulário
lda_stop_words = list(set(lda_stop_words + ["cupom", "investingpro", "protips", "estratégias", "ações"]))

print(lda_topics.invoke({"n_topics": 6, "company": "NVIDIA"}))

In [ ]:
"""Rerun LDA with 8 topics and regenerate all plots without the agent / Refaz o LDA com 8 tópicos e regera todos os gráficos sem o agente."""

# full text result, read the headlines of each topic / resultado completo em texto, leia as manchetes de cada tópico
print(lda_topics.invoke({"n_topics": 8, "company": "NVIDIA"}))

In [ ]:
# regenerate the bar chart and the scatterplot, names are filled after reading the topics / regera o gráfico de barras e o scatterplot, os nomes são preenchidos após ler os tópicos
print(plot_lda_topics.invoke({"topic_names": "{}"}))

In [ ]:
print(plot_lda_scatter.invoke({"topic_names": "{}"}))